In [1]:
# =============================================================================
# SECTION 1: SETUP AND CONFIGURATION
# =============================================================================

import hashlib
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

# Set publication-ready styling
plt.style.use('default')
sns.set_theme(style="whitegrid", palette="husl")
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10
})

print("✅ Imports completed successfully")
print("📊 Publication-ready styling configured")


✅ Imports completed successfully
📊 Publication-ready styling configured


In [2]:
# =============================================================================
# CONFIGURATION CLASS
# =============================================================================

@dataclass(frozen=True)
class Config:
    """
    Configuration class for the research analysis pipeline.
    
    This class contains all parameters used in the data filtering and analysis pipeline.
    All parameters are designed to be easily adjustable for sensitivity analysis.
    """
    # Data source
    RAW: str = "fdv_and_mc_data_raw.csv"
    OUTDIR: str = "outputs"
    
    # Date range
    START: str = "2022-06-01"
    END: str = "2025-06-01"
    
    # Filtering parameters
    MIN_WEEKS: int = 5  # Minimum weeks of data required
    OUTLIER_Q: float = 0.99  # Outlier quantile threshold
    FDV_CAP: float = 1e12  # FDV cap ($1T)
    FDV_VS_MC_MULT: float = 100.0  # FDV vs MC multiplier
    FDV_MISMATCH: float = 1e9  # FDV mismatch threshold
    MC_MISMATCH_FLOOR: float = 1e6  # MC mismatch floor
    CS_MAX_OBS: float = 0.95  # Maximum %CS observed
    CS_DROP: float = 0.10  # %CS drop threshold
    MIN_AVG_MC: float = 5e6  # Minimum average MC ($5M)
    TOP_N: Optional[int] = 1000  # Top N tokens to keep

# Create global configuration instance
CFG = Config()

print("✅ Configuration class defined")
print(f"📊 Analysis parameters: {CFG.MIN_WEEKS} min weeks, {CFG.OUTLIER_Q} outlier threshold")


✅ Configuration class defined
📊 Analysis parameters: 5 min weeks, 0.99 outlier threshold


In [ ]:
# =============================================================================
# ENHANCED HELPER FUNCTIONS WITH TOKEN REMOVAL TRACKING
# =============================================================================

def report_step_enhanced(step_name: str, df: pd.DataFrame, step_number: int, 
                        tokens_before: int = None, tokens_removed: int = None) -> None:
    """
    Enhanced report function that shows token removal statistics.
    
    Args:
        step_name: Name of the filtering step
        df: DataFrame after filtering
        step_number: Step number for display
        tokens_before: Number of tokens before filtering
        tokens_removed: Number of tokens removed in this step
    """
    if tokens_before is not None and tokens_removed is not None:
        print(f"CHECK: Step {step_number}: {len(df):,} rows, {df['symbol'].nunique():,} unique tokens ({tokens_removed:,} tokens removed)")
    else:
        print(f"CHECK: Step {step_number}: {len(df):,} rows, {df['symbol'].nunique():,} unique tokens")

def save_removed_tokens(removed_tokens: set, step_name: str, output_dir: Path) -> None:
    """
    Save list of removed tokens to a text file.
    
    Args:
        removed_tokens: Set of token symbols that were removed
        step_name: Name of the filtering step
        output_dir: Output directory for the file
    """
    if removed_tokens:
        # Create removed tokens directory
        removed_dir = output_dir / "removed_tokens"
        removed_dir.mkdir(exist_ok=True, parents=True)
        
        # Sort tokens for consistent output
        sorted_tokens = sorted(removed_tokens)
        
        # Save to file
        file_path = removed_dir / f"{step_name}.txt"
        with open(file_path, 'w') as f:
            f.write('\n'.join(sorted_tokens))
        
        print(f"  📝 Saved {len(removed_tokens):,} removed tokens to: {file_path}")
    else:
        print(f"  ✅ No tokens removed in step: {step_name}")

def create_removal_summary(removal_tracking: dict, output_dir: Path) -> pd.DataFrame:
    """
    Create a summary DataFrame of token removals at each step.
    
    Args:
        removal_tracking: Dictionary with step names as keys and removal info as values
        output_dir: Output directory for the summary file
        
    Returns:
        DataFrame with removal summary
    """
    summary_data = []
    for step, info in removal_tracking.items():
        summary_data.append({
            'step': step,
            'tokens_before': info['tokens_before'],
            'tokens_after': info['tokens_after'],
            'tokens_removed': info['tokens_removed'],
            'removal_rate': info['tokens_removed'] / info['tokens_before'] if info['tokens_before'] > 0 else 0
        })
    
    summary_df = pd.DataFrame(summary_data)
    
    # Save summary
    summary_path = output_dir / "token_removal_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    
    print(f"📊 Token removal summary saved to: {summary_path}")
    return summary_df

print("✅ Enhanced helper functions with token removal tracking defined")


In [3]:
# =============================================================================
# SECTION 2: CORE PIPELINE FUNCTIONS
# =============================================================================

def load_raw_data(cfg: Config = CFG) -> pd.DataFrame:
    """
    Load and perform initial data preparation.
    
    Args:
        cfg: Configuration object containing data source path
        
    Returns:
        pd.DataFrame: Raw data with basic preprocessing applied
        
    Notes:
        - Renames columns to standard format
        - Computes circulating supply ratio (%CS)
        - Removes invalid entries (FDV=0, negative values, etc.)
    """
    print("📥 Loading raw data...")
    
    # Load data
    df = pd.read_csv(cfg.RAW)
    
    # Rename columns to standard format
    df = df.rename(columns={
        'token_name': 'symbol',
        'fdv': 'FDV',
        'market_cap': 'MC'
    })
    
    # Parse timestamps
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    # Compute circulating supply ratio
    df['%CS'] = df['MC'] / df['FDV']
    
    # Remove invalid entries
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df[df['FDV'] > 0]
    df = df[df['MC'] > 0]
    df = df[df['%CS'].notna() & (df['%CS'] > 0) & (df['%CS'] <= 1)]
    
    print(f"✅ Loaded {len(df):,} rows, {df['symbol'].nunique():,} unique tokens")
    return df

def report_step(step_name: str, df: pd.DataFrame, step_number: int) -> None:
    """
    Report statistics for a filtering step.
    
    Args:
        step_name: Name of the filtering step
        df: DataFrame after filtering
        step_number: Step number for display
    """
    print(f"CHECK: Step {step_number}: {len(df):,} rows, {df['symbol'].nunique():,} unique tokens")

# Step descriptions for the new 10-step sequence
STEP_DESC = {
    "1_raw": "Raw Dataset",
    "2_date_filter": "Date filter (Jun 2022 – Jun 2025)",
    "3_outlier_trim": "Outlier trim (top 1% FDV/MC)",
    "4_quality_guards": "Quality guards (FDV > $1T or > 100× MC)",
    "5_min_weeks": "Min 5 weeks data",
    "6_remove_constant": "Remove constant FDV/%CS",
    "7_cs_95_filter": "%CS < 95 filter",
    "8_remove_cs_drops": "Remove %CS drops > 10%",
    "9_min_avg_mc": "Min avg MC ≥ $5M",
    "10_top_1000_fdv": "Top 1000 by latest FDV",
    "11_final": "Final cleanup and variable creation"
}

print("✅ Core pipeline functions defined")


✅ Core pipeline functions defined


In [ ]:
# =============================================================================
# CLEAN 10-STEP PIPELINE FUNCTION
# =============================================================================

def run_pipeline_with_snapshots(cfg: Config = CFG) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, pd.DataFrame], Dict]:
    """
    Execute the complete data filtering pipeline with snapshots at each step.
    
    Filter Steps:
    1. Raw Dataset
    2. Date filter (Jun 2022 – Jun 2025)
    3. Outlier trim (top 1% FDV/MC)
    4. Quality guards (FDV > $1T or > 100× MC)
    5. Min 5 weeks data
    6. Remove constant FDV/%CS
    7. %CS < 95 filter
    8. Remove %CS drops > 10%
    9. Min avg MC ≥ $5M
    10. Top 1000 by latest FDV
    
    Args:
        cfg: Configuration object
        
    Returns:
        Tuple containing:
        - final_df: Final cleaned dataset
        - cascade_df: Cascade counts at each step
        - snapshots: Dictionary of datasets at each filtering step
        - removal_tracking: Dictionary tracking token removals at each step
    """
    print("🚀 Starting data filtering pipeline...")
    
    # Initialize tracking
    removal_tracking = {}
    output_dir = Path(cfg.OUTDIR)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Step 1: Raw Dataset
    df = load_raw_data(cfg)
    snapshots = {"1_raw": df.copy()}
    tokens_before = df['symbol'].nunique()
    report_step_enhanced("Raw Dataset", df, 1, tokens_before, 0)
    removal_tracking["1_raw"] = {
        "tokens_before": tokens_before,
        "tokens_after": tokens_before,
        "tokens_removed": 0
    }
    
    # Step 2: Date filter (Jun 2022 – Jun 2025)
    tokens_before = df['symbol'].nunique()
    df = df[(df['timestamp'] >= cfg.START) & (df['timestamp'] <= cfg.END)].copy()
    tokens_after = df['symbol'].nunique()
    tokens_removed = tokens_before - tokens_after
    removed_tokens = set(snapshots["1_raw"]['symbol'].unique()) - set(df['symbol'].unique())
    
    snapshots["2_date_filter"] = df.copy()
    report_step_enhanced("Date Filter", df, 2, tokens_before, tokens_removed)
    save_removed_tokens(removed_tokens, "2_date_filter", output_dir)
    removal_tracking["2_date_filter"] = {
        "tokens_before": tokens_before,
        "tokens_after": tokens_after,
        "tokens_removed": tokens_removed
    }
    
    # Step 3: Outlier trim (top 1% FDV/MC)
    tokens_before = df['symbol'].nunique()
    q_fdv = df['FDV'].quantile(cfg.OUTLIER_Q)
    q_mc = df['MC'].quantile(cfg.OUTLIER_Q)
    df = df[(df['FDV'] < q_fdv) & (df['MC'] < q_mc)].copy()
    tokens_after = df['symbol'].nunique()
    tokens_removed = tokens_before - tokens_after
    removed_tokens = set(snapshots["2_date_filter"]['symbol'].unique()) - set(df['symbol'].unique())
    
    snapshots["3_outlier_trim"] = df.copy()
    report_step_enhanced("Outlier Trim", df, 3, tokens_before, tokens_removed)
    save_removed_tokens(removed_tokens, "3_outlier_trim", output_dir)
    removal_tracking["3_outlier_trim"] = {
        "tokens_before": tokens_before,
        "tokens_after": tokens_after,
        "tokens_removed": tokens_removed
    }
    
    # Step 4: Quality guards (FDV > $1T or > 100× MC)
    tokens_before = df['symbol'].nunique()
    df = df[df['FDV'] < cfg.FDV_CAP].copy()  # FDV < $1T
    df = df[df['FDV'] <= df['MC'] * cfg.FDV_VS_MC_MULT].copy()  # FDV <= MC * 100
    df = df[~((df['FDV'] > cfg.FDV_MISMATCH) & (df['MC'] < cfg.MC_MISMATCH_FLOOR))].copy()
    tokens_after = df['symbol'].nunique()
    tokens_removed = tokens_before - tokens_after
    removed_tokens = set(snapshots["3_outlier_trim"]['symbol'].unique()) - set(df['symbol'].unique())
    
    snapshots["4_quality_guards"] = df.copy()
    report_step_enhanced("Quality Guards", df, 4, tokens_before, tokens_removed)
    save_removed_tokens(removed_tokens, "4_quality_guards", output_dir)
    removal_tracking["4_quality_guards"] = {
        "tokens_before": tokens_before,
        "tokens_after": tokens_after,
        "tokens_removed": tokens_removed
    }
    
    # Step 5: Min 5 weeks data
    tokens_before = df['symbol'].nunique()
    token_counts = df['symbol'].value_counts()
    valid_tokens = token_counts[token_counts >= cfg.MIN_WEEKS].index
    df = df[df['symbol'].isin(valid_tokens)].copy()
    tokens_after = df['symbol'].nunique()
    tokens_removed = tokens_before - tokens_after
    removed_tokens = set(snapshots["4_quality_guards"]['symbol'].unique()) - set(df['symbol'].unique())
    
    snapshots["5_min_weeks"] = df.copy()
    report_step_enhanced("Min 5 Weeks Data", df, 5, tokens_before, tokens_removed)
    save_removed_tokens(removed_tokens, "5_min_weeks", output_dir)
    removal_tracking["5_min_weeks"] = {
        "tokens_before": tokens_before,
        "tokens_after": tokens_after,
        "tokens_removed": tokens_removed
    }
    
    # Step 6: Remove constant FDV/%CS
    tokens_before = df['symbol'].nunique()
    fdv_std = df.groupby('symbol')['FDV'].std()
    cs_std = df.groupby('symbol')['%CS'].std()
    valid_tokens = fdv_std[(fdv_std > 0) & (cs_std > 0)].index
    df = df[df['symbol'].isin(valid_tokens)].copy()
    tokens_after = df['symbol'].nunique()
    tokens_removed = tokens_before - tokens_after
    removed_tokens = set(snapshots["5_min_weeks"]['symbol'].unique()) - set(df['symbol'].unique())
    
    snapshots["6_remove_constant"] = df.copy()
    report_step_enhanced("Remove Constant FDV/%CS", df, 6, tokens_before, tokens_removed)
    save_removed_tokens(removed_tokens, "6_remove_constant", output_dir)
    removal_tracking["6_remove_constant"] = {
        "tokens_before": tokens_before,
        "tokens_after": tokens_after,
        "tokens_removed": tokens_removed
    }
    
    # Step 7: %CS < 95 filter
    tokens_before = df['symbol'].nunique()
    cs_max = df.groupby('symbol')['%CS'].max()
    valid_tokens = cs_max[cs_max < cfg.CS_MAX_OBS].index
    df = df[df['symbol'].isin(valid_tokens)].copy()
    tokens_after = df['symbol'].nunique()
    tokens_removed = tokens_before - tokens_after
    removed_tokens = set(snapshots["6_remove_constant"]['symbol'].unique()) - set(df['symbol'].unique())
    
    snapshots["7_cs_95_filter"] = df.copy()
    report_step_enhanced("%CS < 95 Filter", df, 7, tokens_before, tokens_removed)
    save_removed_tokens(removed_tokens, "7_cs_95_filter", output_dir)
    removal_tracking["7_cs_95_filter"] = {
        "tokens_before": tokens_before,
        "tokens_after": tokens_after,
        "tokens_removed": tokens_removed
    }
    
    # Step 8: Remove %CS drops > 10%
    tokens_before = df['symbol'].nunique()
    df_sorted = df.sort_values(['symbol', 'timestamp']).copy()
    df_sorted['cs_diff'] = df_sorted.groupby('symbol')['%CS'].diff()
    tokens_with_drops = df_sorted[df_sorted['cs_diff'] < -cfg.CS_DROP]['symbol'].unique()
    df = df[~df['symbol'].isin(tokens_with_drops)].copy()
    tokens_after = df['symbol'].nunique()
    tokens_removed = tokens_before - tokens_after
    removed_tokens = set(snapshots["7_cs_95_filter"]['symbol'].unique()) - set(df['symbol'].unique())
    
    snapshots["8_remove_cs_drops"] = df.copy()
    report_step_enhanced("Remove %CS Drops > 10%", df, 8, tokens_before, tokens_removed)
    save_removed_tokens(removed_tokens, "8_remove_cs_drops", output_dir)
    removal_tracking["8_remove_cs_drops"] = {
        "tokens_before": tokens_before,
        "tokens_after": tokens_after,
        "tokens_removed": tokens_removed
    }
    
    # Step 9: Min avg MC ≥ $5M
    tokens_before = df['symbol'].nunique()
    avg_mc = df.groupby('symbol')['MC'].mean()
    valid_tokens = avg_mc[avg_mc >= cfg.MIN_AVG_MC].index
    df = df[df['symbol'].isin(valid_tokens)].copy()
    tokens_after = df['symbol'].nunique()
    tokens_removed = tokens_before - tokens_after
    removed_tokens = set(snapshots["8_remove_cs_drops"]['symbol'].unique()) - set(df['symbol'].unique())
    
    snapshots["9_min_avg_mc"] = df.copy()
    report_step_enhanced("Min Avg MC ≥ $5M", df, 9, tokens_before, tokens_removed)
    save_removed_tokens(removed_tokens, "9_min_avg_mc", output_dir)
    removal_tracking["9_min_avg_mc"] = {
        "tokens_before": tokens_before,
        "tokens_after": tokens_after,
        "tokens_removed": tokens_removed
    }
    
    # Step 10: Top 1000 by latest FDV
    tokens_before = df['symbol'].nunique()
    if cfg.TOP_N is not None:
        latest_date = df['timestamp'].max()
        top_tokens = (df[df['timestamp'] == latest_date]
                     .groupby('symbol')['FDV']
                     .mean()
                     .sort_values(ascending=False)
                     .head(cfg.TOP_N)
                     .index)
        df = df[df['symbol'].isin(top_tokens)].copy()
        tokens_after = df['symbol'].nunique()
        tokens_removed = tokens_before - tokens_after
        removed_tokens = set(snapshots["9_min_avg_mc"]['symbol'].unique()) - set(df['symbol'].unique())
        
        snapshots["10_top_1000_fdv"] = df.copy()
        report_step_enhanced("Top 1000 by Latest FDV", df, 10, tokens_before, tokens_removed)
        save_removed_tokens(removed_tokens, "10_top_1000_fdv", output_dir)
        removal_tracking["10_top_1000_fdv"] = {
            "tokens_before": tokens_before,
            "tokens_after": tokens_after,
            "tokens_removed": tokens_removed
        }
    else:
        snapshots["10_top_1000_fdv"] = df.copy()
        report_step_enhanced("Top 1000 by Latest FDV (skipped)", df, 10, tokens_before, 0)
        removal_tracking["10_top_1000_fdv"] = {
            "tokens_before": tokens_before,
            "tokens_after": tokens_before,
            "tokens_removed": 0
        }
    
    # Final cleanup and variable creation
    tokens_before = df['symbol'].nunique()
    df['TGE_date'] = df.groupby('symbol')['timestamp'].transform('min')
    df['day_from_TGE'] = (df['timestamp'] - df['TGE_date']).dt.days
    df.drop(columns=['TGE_date'], inplace=True)
    
    df = df.drop_duplicates(subset=['timestamp', 'symbol']).sort_values(['symbol', 'timestamp'])
    df = df.reset_index(drop=True)
    df = df[['timestamp', 'symbol', 'day_from_TGE', '%CS', 'MC', 'FDV']]
    
    snapshots["11_final"] = df.copy()
    report_step_enhanced("Final Dataset", df, 11, tokens_before, 0)
    removal_tracking["11_final"] = {
        "tokens_before": tokens_before,
        "tokens_after": tokens_before,
        "tokens_removed": 0
    }
    
    # Create cascade counts
    cascade_data = []
    for step, data in snapshots.items():
        cascade_data.append({
            'step': step,
            'tokens': data['symbol'].nunique(),
            'rows': len(data),
            'start': data['timestamp'].min() if len(data) > 0 else None,
            'end': data['timestamp'].max() if len(data) > 0 else None
        })
    
    cascade_df = pd.DataFrame(cascade_data)
    
    # Create removal summary
    removal_summary = create_removal_summary(removal_tracking, output_dir)
    
    print(f"✅ Pipeline completed: {len(df):,} rows, {df['symbol'].nunique():,} unique tokens")
    return df, cascade_df, snapshots, removal_tracking

print("✅ Clean 10-step pipeline function defined")


✅ Clean 10-step pipeline function defined


In [5]:
# =============================================================================
# ANALYSIS HELPER FUNCTIONS
# =============================================================================

def ensure_vars(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure required variables exist for regression analysis.
    
    Args:
        df: DataFrame to process
        
    Returns:
        DataFrame with log_MC, log_FDV, and cs_ratio variables
    """
    df = df.copy()
    
    # Create log variables
    df['log_MC'] = np.log(df['MC'])
    df['log_FDV'] = np.log(df['FDV'])
    df['cs_ratio'] = df['%CS']
    
    # Remove any infinite or NaN values
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=['log_MC', 'log_FDV', 'cs_ratio'])
    
    return df

def fit_cluster_ols(y_var: str, df: pd.DataFrame) -> Dict:
    """
    Fit cluster-robust OLS regression.
    
    Args:
        y_var: Dependent variable name
        df: DataFrame with data
        
    Returns:
        Dictionary with regression results
    """
    try:
        # Prepare data
        reg_data = ensure_vars(df)
        
        if len(reg_data) == 0:
            return {"coef": np.nan, "p": np.nan, "R2": np.nan, "N": 0}
        
        # Fit OLS regression
        formula = f"{y_var} ~ cs_ratio"
        model = smf.ols(formula, data=reg_data).fit()
        
        # Get cluster-robust standard errors
        cluster_groups = reg_data['symbol']
        
        # Calculate cluster-robust standard errors
        try:
            # Use statsmodels cluster-robust standard errors
            from statsmodels.stats.sandwich_covariance import cov_cluster
            
            # Get residuals and design matrix
            exog = model.model.exog
            endog = model.model.endog
            
            # Calculate cluster-robust covariance
            cov_cluster_robust = cov_cluster(model, cluster_groups)
            
            # Get t-statistic and p-value for cs_ratio coefficient
            coef_idx = 1  # cs_ratio coefficient index
            se_cluster = np.sqrt(cov_cluster_robust[coef_idx, coef_idx])
            t_stat = model.params.iloc[coef_idx] / se_cluster
            p_value = 2 * (1 - stats.norm.cdf(abs(t_stat)))
            
        except:
            # Fallback to regular standard errors
            se_cluster = model.bse.iloc[1]
            t_stat = model.tvalues.iloc[1]
            p_value = model.pvalues.iloc[1]
        
        return {
            "coef": model.params.iloc[1],  # cs_ratio coefficient
            "p": p_value,
            "R2": model.rsquared,
            "N": len(reg_data)
        }
        
    except Exception as e:
        print(f"Warning: Regression failed for {y_var}: {e}")
        return {"coef": np.nan, "p": np.nan, "R2": np.nan, "N": 0}

def order_steps(df: pd.DataFrame) -> pd.DataFrame:
    """
    Order steps for consistent analysis.
    
    Args:
        df: DataFrame with step results
        
    Returns:
        DataFrame ordered by step sequence
    """
    desired_order = [
        "1_raw", "2_date_filter", "3_outlier_trim", "4_quality_guards", 
        "5_min_weeks", "6_remove_constant", "7_cs_95_filter", 
        "8_remove_cs_drops", "9_min_avg_mc", "10_top_1000_fdv", "11_final"
    ]
    
    # Create mapping for ordering
    step_order = {step: i for i, step in enumerate(desired_order)}
    df['step_order'] = df['step'].map(step_order)
    
    # Sort by step order and drop the ordering column
    df = df.sort_values('step_order').drop('step_order', axis=1)
    
    return df

print("✅ Analysis helper functions defined")


✅ Analysis helper functions defined


In [ ]:
# =============================================================================
# MAIN ANALYSIS FUNCTION
# =============================================================================

def analyse_step_impacts(cfg: Config = CFG) -> Dict:
    """
    Analyze the impact of each filtering step on regression relationships.
    
    This function runs the complete pipeline and performs regression analysis
    at each step to understand how filtering affects the relationships between
    %CS, log(MC), and log(FDV).
    
    Args:
        cfg: Configuration object
        
    Returns:
        Dict containing final dataset, cascade data, impact analysis, and snapshots
    """
    print("🔬 Starting comprehensive step impact analysis...")
    
    # Run the pipeline with snapshots
    final_df, cascade_df, snapshots, removal_tracking = run_pipeline_with_snapshots(cfg)
    
    # Create output directories
    base_dir = Path(cfg.OUTDIR) / "step_impact"
    base_dir.mkdir(parents=True, exist_ok=True)
    
    # Initialize impact analysis
    impact_rows = []
    
    print("\n📊 Performing regression analysis at each filtering step...")
    
    # Analyze each step
    for step_key, step_data in snapshots.items():
        print(f"  Analyzing step: {step_key}")
        
        # Create step directory
        step_dir = base_dir / step_key
        step_dir.mkdir(exist_ok=True, parents=True)
        
        # Prepare data for regression
        reg_data = ensure_vars(step_data)
        
        # Fit regressions
        mc_results = fit_cluster_ols("log_MC", reg_data)
        fdv_results = fit_cluster_ols("log_FDV", reg_data)
        
        # Store results
        impact_rows.append({
            "step": step_key,
            "tokens": int(step_data["symbol"].nunique()),
            "rows": int(len(step_data)),
            "coef_CS_logMC": mc_results["coef"],
            "p_CS_logMC": mc_results["p"],
            "R2_logMC": mc_results["R2"],
            "N_logMC": mc_results["N"],
            "coef_CS_logFDV": fdv_results["coef"],
            "p_CS_logFDV": fdv_results["p"],
            "R2_logFDV": fdv_results["R2"],
            "N_logFDV": fdv_results["N"],
            "clusters": int(step_data["symbol"].nunique())
        })
        
        # Save step manifest
        manifest = {
            "step": step_key,
            "tokens": int(step_data["symbol"].nunique()),
            "rows": int(len(step_data)),
            "start": str(step_data["timestamp"].min()) if len(step_data) > 0 else None,
            "end": str(step_data["timestamp"].max()) if len(step_data) > 0 else None
        }
        
        with open(step_dir / "manifest.json", "w") as f:
            json.dump(manifest, f, indent=2)
    
    # Create impact summary
    impact_df = pd.DataFrame(impact_rows)
    impact_df = order_steps(impact_df)
    
    # Save results
    impact_path = base_dir / "step_impact_summary.csv"
    impact_df.to_csv(impact_path, index=False)
    cascade_df.to_csv(base_dir / "cascade_counts.csv", index=False)
    
    # Save run manifest
    run_manifest = {
        "timestamp": pd.Timestamp.now().isoformat(),
        "config": {
            "START": cfg.START,
            "END": cfg.END,
            "MIN_WEEKS": cfg.MIN_WEEKS,
            "OUTLIER_Q": cfg.OUTLIER_Q,
            "FDV_CAP": cfg.FDV_CAP,
            "CS_MAX_OBS": cfg.CS_MAX_OBS,
            "MIN_AVG_MC": cfg.MIN_AVG_MC,
            "TOP_N": cfg.TOP_N
        },
        "final_stats": {
            "tokens": int(final_df["symbol"].nunique()),
            "rows": int(len(final_df)),
            "start": str(final_df["timestamp"].min()),
            "end": str(final_df["timestamp"].max())
        }
    }
    
    with open(Path(cfg.OUTDIR) / "run_manifest.json", "w") as f:
        json.dump(run_manifest, f, indent=2)
    
    print(f"\n✅ Step impact analysis completed")
    print(f"📁 Results saved to: {base_dir.resolve()}")
    print(f"📊 Impact summary: {impact_path}")
    
    return {
        "final_df": final_df,
        "cascade": cascade_df,
        "impact": impact_df,
        "snapshots": snapshots,
        "removal_tracking": removal_tracking
    }

print("✅ Main analysis function defined")


✅ Main analysis function defined


In [ ]:
# =============================================================================
# EXECUTE CLEAN ANALYSIS
# =============================================================================

print("🚀 Starting clean analysis with 10-step filter sequence...")
print("=" * 80)

# Execute the step impact analysis
results = analyse_step_impacts(CFG)

# Extract results
final_df = results["final_df"]
cascade_df = results["cascade"] 
impact_df = results["impact"]
snapshots = results["snapshots"]
removal_tracking = results["removal_tracking"]

print("\n" + "=" * 80)
print("📊 CLEAN ANALYSIS RESULTS SUMMARY")
print("=" * 80)

print(f"Final dataset: {len(final_df):,} rows, {final_df['symbol'].nunique():,} unique tokens")
print(f"Date range: {final_df['timestamp'].min()} to {final_df['timestamp'].max()}")
print(f"Analysis completed for {len(impact_df)} filtering steps")

print("\n📈 Step Impact Analysis Results:")
print(impact_df[['step', 'tokens', 'rows', 'coef_CS_logMC', 'p_CS_logMC', 'R2_logMC', 
                'coef_CS_logFDV', 'p_CS_logFDV', 'R2_logFDV']].round(4))

print("\n📊 Token Removal Summary:")
removal_summary = pd.DataFrame([
    {
        'step': step,
        'tokens_before': info['tokens_before'],
        'tokens_after': info['tokens_after'],
        'tokens_removed': info['tokens_removed'],
        'removal_rate': f"{info['tokens_removed'] / info['tokens_before'] * 100:.1f}%" if info['tokens_before'] > 0 else "0.0%"
    }
    for step, info in removal_tracking.items()
])
print(removal_summary)

print("\n✅ Clean analysis execution completed successfully!")
print("🎯 Now using the correct 10-step filter sequence!")
print("📁 Token removal files saved to: outputs/removed_tokens/")
print("📊 Token removal summary saved to: outputs/token_removal_summary.csv")


🚀 Starting clean analysis with 10-step filter sequence...
🔬 Starting comprehensive step impact analysis...
🚀 Starting data filtering pipeline...
📥 Loading raw data...
✅ Loaded 261,567 rows, 3,962 unique tokens
CHECK: Step 1: 261,567 rows, 3,962 unique tokens
CHECK: Step 2: 229,626 rows, 3,816 unique tokens
CHECK: Step 3: 225,708 rows, 3,786 unique tokens
CHECK: Step 4: 221,629 rows, 3,744 unique tokens
CHECK: Step 5: 221,108 rows, 3,500 unique tokens
CHECK: Step 6: 178,438 rows, 2,812 unique tokens
CHECK: Step 7: 138,377 rows, 1,867 unique tokens
CHECK: Step 8: 114,003 rows, 1,598 unique tokens
CHECK: Step 9: 68,770 rows, 874 unique tokens
CHECK: Step 10: 66,253 rows, 826 unique tokens
CHECK: Step 11: 66,253 rows, 826 unique tokens
✅ Pipeline completed: 66,253 rows, 826 unique tokens

📊 Performing regression analysis at each filtering step...
  Analyzing step: 1_raw
  Analyzing step: 2_date_filter
  Analyzing step: 3_outlier_trim
  Analyzing step: 4_quality_guards
  Analyzing step: 5_m